# 0. librerias 

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("../../data/NovaMarket_datos_crudos.csv")

In [2]:
df.head()

,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,unidades_vendidas,monto_compra,edad_cliente,correo_cliente,nivel_satisfaccion,nivel_lealtad,comentario_cliente,peso_pedido_kg,fecha_actualizacion_stock,fuga_cliente
0,P505195,C45650,15/11/2025,Tienda,Efectivo,bogota,1100103,Moda,Chaqueta impermeable,28700.0,1,28700.0,32,cliente405195@novamarket.com.co,Bajo,Platino,El domiciliario fue muy amable,14.9,2026-04-17,0
1,P677836,C84260,2026-03-05,Web,Efectivo,MZL,1700101,Electrónica,Parlante bluetooth,195700.0,2,391400.0,73,cliente577836@yahoo.es,Medio,Bronce,El producto llegó dañado,12.9,2026-06-10,0
2,P387869,C164166,2025-03-13,Tienda,PayPal,Cúcuta,5400101,Juguetería,Rompecabezas 1000 piezas,597600.0,-3,-1792800.0,32,cliente287869@novamarket.com.co,Bajo,Oro,El domiciliario fue muy amable,14.3,2026-02-26,0
3,P481069,C98406,27/06/2025,App,transferencia,Cucuta,5400103,deportes,Colchoneta de yoga,322900.0,1,322900.0,45,cliente381069@gmail.com,Medio,Platino,Rápido y sin problemas,9.5,2026-05-14,0
4,P247920,C163018,01/05/2025,Web,Efectivo,Bucaramanga,6800112,Hogar,Juego de sábanas,222400.0,1,222400.0,60,cliente147920@novamarket.com.co,Alto,Plata,NaN,12.3,2026-07-13,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 640000 entries, 0 to 639999
Data columns (total 20 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   id_pedido                  640000 non-null  str    
 1   id_cliente                 640000 non-null  str    
 2   fecha_pedido               640000 non-null  str    
 3   canal_compra               640000 non-null  str    
 4   metodo_pago                640000 non-null  str    
 5   ciudad_tienda              610072 non-null  str    
 6   codigo_postal              640000 non-null  int64  
 7   categoria_producto         640000 non-null  str    
 8   producto                   640000 non-null  str    
 9   precio_unitario            618360 non-null  float64
 10  unidades_vendidas          640000 non-null  int64  
 11  monto_compra               618360 non-null  float64
 12  edad_cliente               640000 non-null  int64  
 13  correo_cliente             534688 non-nu

In [4]:
df.isna().sum()

id_pedido                         0
id_cliente                        0
fecha_pedido                      0
canal_compra                      0
metodo_pago                       0
ciudad_tienda                 29928
codigo_postal                     0
categoria_producto                0
producto                          0
precio_unitario               21640
unidades_vendidas                 0
monto_compra                  21640
edad_cliente                      0
correo_cliente               105312
nivel_satisfaccion            46469
nivel_lealtad                     0
comentario_cliente            90806
peso_pedido_kg                    0
fecha_actualizacion_stock         0
fuga_cliente                      0
dtype: int64

# 1. Mcar 

## precio_unitario: 3.40% datos faltantes

**Prioridad:** Alta | **Semana:** 4 | **Mecanismo:** MCAR (Missing Completely At Random)

**Diagnóstico:** Se comparó la proporción de `categoria_producto` en las 
filas con `precio_unitario` nulo contra la proporción en todo el dataset, 
y las proporciones se mantienen prácticamente iguales. Esto descarta que 
la ausencia dependa de otra variable observada, confirmando MCAR.

**Estrategia:** Eliminación de filas.

**Justificación:** Al ser MCAR y estar por debajo del umbral del 5%, 
eliminar no introduce sesgo ni distorsiona la distribución. `precio_unitario` 
es un campo transaccional core que afecta cálculos financieros directos 
(ingresos, márgenes), por lo que su integridad es crítica para el negocio 
— de ahí la prioridad alta.

**Nota de dependencia:** esta eliminación debe ejecutarse **antes** del 
recálculo de `monto_compra` (que depende de `precio_unitario`), para que 
el recálculo no arrastre filas con precio nulo.

In [5]:
df = df.dropna(subset=["precio_unitario"])

In [6]:
print("precio_unitario nulos:", df["precio_unitario"].isna().sum())
print("filas restantes:", df.shape[0])

assert df["precio_unitario"].isna().sum() == 0, "Todavía quedan nulos en precio_unitario"

precio_unitario nulos: 0
filas restantes: 618360


## monto_compra: 3.40% datos faltantes + 1.10% falsos completos

**Diagnóstico:** La tabla cruzada contra `precio_unitario` mostró una 
correspondencia exacta 1 a 1 (100% de los nulos de `monto_compra` coinciden 
con nulos de `precio_unitario`), confirmando que no es un valor 
independiente sino una dependencia matemática directa: 
`monto_compra = precio_unitario × unidades_vendidas`.

Adicionalmente, un 1.10% de los datos aparentaba estar completo pero tenía 
formato de texto no numérico (símbolos, separadores mal interpretados), 
ocultando su verdadero estado de faltante — un "falso completo" más 
riesgoso que un nulo explícito, porque puede pasar desapercibido.

**Estrategia:** 
1. Convertir a numérico con `pd.to_numeric(errors="coerce")` para revelar 
   los falsos completos como nulos reales.
2. Recalcular `monto_compra = precio_unitario × unidades_vendidas` 
   **únicamente en las filas donde quedó nulo** tras la conversión (no se 
   sobrescribe todo el dataset, solo se completa lo faltante).

**Justificación:** No se debe imputar `monto_compra` con una mediana por 
grupo, porque eso "inventaría" un valor cuando en realidad se puede derivar 
matemáticamente. Mientras el dato esté mal tipado o ausente, es inutilizable 
para cálculos financieros de ingresos y márgenes — de ahí la prioridad alta.

**Nota de dependencia:** este paso debe ejecutarse **después** de tratar 
los nulos de `precio_unitario` (eliminación de filas), para que la fórmula 
de recálculo no arrastre valores nulos de esa columna.

In [7]:
# Solo limpiar formato de texto si la columna llegó como texto (object)
if df["monto_compra"].dtype == "object":
    df["monto_compra"] = (
        df["monto_compra"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace("nan", pd.NA)
    )

# Esto es seguro siempre: si ya es numérico, no hace nada; si quedó texto raro, lo vuelve NaN
df["monto_compra"] = pd.to_numeric(df["monto_compra"], errors="coerce")

# Recalcular SOLO donde monto_compra es nulo
df["monto_compra"] = df["monto_compra"].fillna(
    df["precio_unitario"] * df["unidades_vendidas"]
)

print(df["monto_compra"].isna().sum())

0


In [8]:
# Verifica que los valores recalculados coincidan con la fórmula
inconsistentes = df[
    ~np.isclose(df["monto_compra"], df["precio_unitario"] * df["unidades_vendidas"])
]
print("Filas donde monto_compra no coincide con precio_unitario × unidades_vendidas:", len(inconsistentes))

Filas donde monto_compra no coincide con precio_unitario × unidades_vendidas: 0


## ciudad_tienda: 4.70% datos faltantes

**Diagnóstico:** Se comparó la tasa de faltantes por `categoria_producto` y 
por `canal_compra`, y las proporciones se mantienen parejas entre grupos 
(sin ningún grupo que se dispare), descartando MAR. El bajo porcentaje y 
la ausencia de patrón confirman MCAR.

**Estrategia:** Eliminación de filas.

**Justificación:** MCAR y bajo porcentaje permiten eliminar sin pérdida 
significativa de información. `ciudad_tienda` es un campo 
descriptivo/geográfico, útil para segmentación pero no crítico para 
cálculos financieros — de ahí la prioridad media.

In [9]:
df = df.dropna(subset=["ciudad_tienda"])

In [10]:
print("ciudad_tienda nulos:", df["ciudad_tienda"].isna().sum())
print("filas restantes:", df.shape[0])

assert df["ciudad_tienda"].isna().sum() == 0, "Todavía quedan nulos en ciudad_tienda"

ciudad_tienda nulos: 0
filas restantes: 589433


# MAR


## nivel_satisfaccion: 7.60% datos faltantes

**Prioridad:** Media | **Semana:** 4 | **Mecanismo:** MAR (Missing At Random)

**Diagnóstico:** El canal "app" muestra 19.9%-21.0% de faltantes en las seis 
categorías de producto, frente a ~3% en los demás canales, aislando a 
`canal_compra` como variable explicativa. Esto confirma MAR: el dato no 
falta al azar, sino que depende de una variable observada (el canal).

**Estrategia:** Imputación por grupo según `canal_compra` (moda por canal).

**Justificación:** Al ser MAR, imputar con la moda calculada dentro de cada 
canal respeta el patrón detectado, en vez de usar una moda global que 
estaría sesgada hacia el canal "app" (que concentra la mayoría de los 
faltantes). Esta imputación requiere que `canal_compra` ya esté normalizado 
(sin variantes de mayúsculas/espacios), para que los grupos sean correctos.

**Prioridad media:** afecta la experiencia de cliente, no cálculos 
financieros core. Se recomienda reportar al área de producto el problema 
de captura de este dato en la app.

#### canal_compra: 51.85% inconsistencia en nombres

**Estrategia:** Normalización de texto (`.strip().lower()`, sin tildes) y 
consolidación a las 4 categorías reales: app, marketplace, tienda, web.

**Justificación:** Tras normalizar, las 331,845 filas se consolidan sin 
pérdida en 4 categorías reales con distribución pareja (~160,000 cada una), 
confirmando que es solo un problema de formato, no de datos distintos. 
canal_compra es variable base para segmentar prácticamente todo el 
análisis (ya usada en diagnósticos MAR de otras columnas).

In [11]:
# Antes de normalizar (guarda copia para comparar)
canales_antes = df["canal_compra"].nunique()
valores_antes = df["canal_compra"].value_counts()

print(f"Canales distintos ANTES: {canales_antes}")
print(valores_antes)

Canales distintos ANTES: 20
canal_compra
Marketplace     111927
Tienda          111796
Web             111766
App             111495
tienda           14450
web              14286
app              14247
marketplace      14159
APP               7188
Marketplace       7161
 Tienda           7159
 Web              7159
 Marketplace      7152
TIENDA            7103
Web               7095
MARKETPLACE       7091
App               7079
WEB               7066
Tienda            7040
 App              7014
Name: count, dtype: int64


In [12]:
df["canal_compra"] = (
    df["canal_compra"]
    .str.strip()
    .str.lower()
    .replace({
        "app": "App",
        "marketplace": "Marketplace",
        "tienda": "Tienda",
        "web": "Web"
    })
)

In [13]:
# Después de normalizar
canales_despues = df["canal_compra"].nunique()
valores_despues = df["canal_compra"].value_counts()

print(f"Canales distintos DESPUÉS: {canales_despues}")
print(valores_despues)

assert canales_despues == 4, "Todavía hay variantes sin consolidar"

Canales distintos DESPUÉS: 4
canal_compra
Tienda         147548
Marketplace    147490
Web            147372
App            147023
Name: count, dtype: int64


In [14]:
moda_satisfaccion = (
    df.groupby("canal_compra")["nivel_satisfaccion"]
    .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

df["nivel_satisfaccion"] = df["nivel_satisfaccion"].fillna(moda_satisfaccion)

In [15]:
print("nivel_satisfaccion nulos ANTES esperado:", "debiste ver un valor > 0 antes de correr el fillna")
print("nivel_satisfaccion nulos DESPUÉS:", df["nivel_satisfaccion"].isna().sum())

# Verifica que la moda se calculó por canal y no globalmente
print(df.groupby("canal_compra")["nivel_satisfaccion"].agg(lambda x: x.mode().iloc[0]))

nivel_satisfaccion nulos ANTES esperado: debiste ver un valor > 0 antes de correr el fillna
nivel_satisfaccion nulos DESPUÉS: 0
canal_compra
App            Bajo
Marketplace    Alto
Tienda         Alto
Web            Alto
Name: nivel_satisfaccion, dtype: str


# MNAR

## correo_cliente: 16.40% datos faltantes

**Prioridad:** Media | **Semana:** 4 | **Mecanismo:** MNAR sospechado (Missing Not At Random)

**Diagnóstico:** Se probó la tasa de faltantes por `categoria_producto` 
(16.0%-16.9%), `canal_compra` (15.7%-17.6%), `metodo_pago` y 
`nivel_lealtad`, todas parejas sin grupos que se disparen — esto descarta 
MAR. Se concluye MNAR sospechado: el cliente decide no dar su correo (no 
es obligatorio en el checkout), y esa decisión está ligada al valor mismo 
que falta.

**Estrategia:** Columna indicadora, sin imputar.

**Justificación:** No se imputa porque inventaría un dato de contacto 
falso. Se conserva el faltante y se agrega una columna indicadora 
(`correo_cliente_faltante`) para dejar registro de este patrón y que 
pueda usarse en análisis posteriores (por ejemplo, para ver si la falta 
de correo se relaciona con otras variables como fuga de cliente). 
Prioridad media: afecta contacto/marketing, no cálculos financieros core.

In [16]:
df["correo_cliente_faltante"] = df["correo_cliente"].isna().astype(int)

In [17]:
print("correo_cliente nulos:", df["correo_cliente"].isna().sum())
print("correo_cliente_faltante == 1:", (df["correo_cliente_faltante"] == 1).sum())

# Deben coincidir exactamente
assert df["correo_cliente"].isna().sum() == (df["correo_cliente_faltante"] == 1).sum(), \
    "La columna indicadora no coincide con los nulos reales"

correo_cliente nulos: 96894
correo_cliente_faltante == 1: 96894


## Comentario_cliente: 14.20% datos faltantes

**Prioridad:** Baja | **Semana:** 4 | **Mecanismo:** MNAR sospechado (Missing Not At Random)

**Diagnóstico:** Se probó la tasa de faltantes por `categoria_producto`, 
`canal_compra`, `metodo_pago` y `nivel_lealtad` (todas ~14.0%-14.3%, sin 
grupos que se disparen), descartando MAR. Se concluye MNAR sospechado: 
dejar comentario es una acción voluntaria del cliente, ligada al valor 
mismo que falta (quien no quiere comentar, no lo hace, y por eso el dato 
falta).

**Estrategia:** Columna indicadora, sin imputar.

**Justificación:** No es apropiado inventar una opinión falsa. Se conserva 
el faltante y se agrega una columna indicadora 
(`comentario_cliente_faltante`) como señal para análisis posterior. 
Prioridad baja: es el campo con menor impacto directo en el negocio de 
los seis que se trataron esta semana.

In [18]:
df["comentario_cliente_faltante"] = df["comentario_cliente"].isna().astype(int)

In [19]:
print("comentario_cliente nulos:", df["comentario_cliente"].isna().sum())
print("comentario_cliente_faltante == 1:", (df["comentario_cliente_faltante"] == 1).sum())

assert df["comentario_cliente"].isna().sum() == (df["comentario_cliente_faltante"] == 1).sum(), \
    "La columna indicadora no coincide con los nulos reales"

comentario_cliente nulos: 83630
comentario_cliente_faltante == 1: 83630


# Generar csv 

In [20]:
df.to_csv(
    "S04_PD_Lozano_DatasetImputacionInicial.csv",
    index=False
)